💡 **Environment:** `clamp-analyses` (R kernel)


# Supplementary Figure 6: reproducibility of CLAMPfull and CLAMPbase modules across independent runs

One 55 × 70 mm panel. For each of the ten ARCHS4 runs, the share of its modules that pass the technical-noise screen and
fall in an MGM, a cluster with members from at least `min_models` runs (`workflow/config/consensus.yaml`). Inputs are the
CLAMPfull clustering (`crispercas_metalv_cluster`) and the CLAMPbase clustering (`consensus_base_cluster`). Labels give the
pooled share; the bracket is a two-sided paired exact Wilcoxon test across runs.


In [ ]:
source(here::here("scripts/panels/main_figure.R"))
set.seed(1)
dirs <- main_figure_dirs("supp6")
min_models <- yaml::read_yaml(here("workflow/config/consensus.yaml"))$consensus$min_models
inputs <- c(base = here("output/98_final_models/consensus_models_base_ge4r015med/archs4/all_lv_clusters.csv"),
            full = here("output/03_model_biology/02_archs4/04_crispercas_ge4r015med/00_mgm/all_lv_clusters.csv"))
lv <- c(base = "ARCHS4\nCLAMPbase", full = "ARCHS4\nCLAMPfull")
cols <- setNames(MODEL_COLORS[c("CLAMPbase", "CLAMPfull")], lv)
lvs <- rbindlist(lapply(names(inputs), function(g) fread(inputs[[g]])[, group := g]))
lvs[, in_mgm := uniqueN(model) >= min_models, by = .(group, cluster)]
seed <- lvs[, .(frac = mean(in_mgm), n = .N), by = .(model, group)]
pool <- lvs[, .(k = sum(in_mgm), N = .N), by = group]
seed[, group := factor(lv[group], levels = lv)]
pool[, group := factor(lv[group], levels = lv)]
stopifnot(seed[, .N, by = group]$N == 10L)
wide <- dcast(seed, model ~ group, value.var = "frac")
test <- data.table(p = wilcox.test(wide[[lv[["full"]]]], wide[[lv[["base"]]]], paired = TRUE, exact = TRUE)$p.value)
lab <- merge(pool, seed[, .(vmax = max(frac)), by = group], by = "group")
fwrite(seed, file.path(dirs$source, "per_run_mgm_fraction.csv"))
fwrite(pool, file.path(dirs$source, "pooled_mgm_fraction.csv"))
fwrite(test, file.path(dirs$source, "paired_test.csv"))
top <- max(seed$frac)
y <- top + 0.11
fig <- ggplot(seed, aes(group, frac, fill = group)) +
  geom_violin(width = 0.75, trim = TRUE, scale = "width", colour = "black", linewidth = 0.22, alpha = 0.65) +
  geom_boxplot(width = 0.18, outlier.shape = NA, colour = "black", linewidth = 0.22, alpha = 0.85) +
  geom_jitter(width = 0.12, height = 0, size = 0.8, shape = 21, fill = "white", colour = "#333333", stroke = 0.25) +
  geom_text(data = lab, aes(group, vmax + 0.045, label = sprintf("%.0f%%", 100 * k / N)), vjust = 0, size = 5.5 / .pt,
            fontface = "bold", inherit.aes = FALSE) +
  geom_text(data = lab, aes(group, vmax + 0.02, label = sprintf("%s / %s", format(k, big.mark = ","), format(N, big.mark = ","))),
            vjust = 0, size = 5 / .pt, inherit.aes = FALSE) +
  annotate("segment", x = 1, xend = 2, y = y, yend = y, linewidth = 0.22) +
  annotate("segment", x = c(1, 2), xend = c(1, 2), y = y - 0.012, yend = y, linewidth = 0.22) +
  annotate("text", x = 1.5, y = y + 0.006, label = sprintf("italic(P) == %.3f", test$p), parse = TRUE, size = 5 / .pt, vjust = 0) +
  scale_fill_manual(values = cols, guide = "none") +
  scale_y_continuous(labels = scales::percent, breaks = seq(0, 0.75, 0.25)) +
  coord_cartesian(ylim = c(0, top + 0.18)) +
  labs(x = NULL, y = "% of modules in MGMs") +
  theme_nm() + theme(axis.text.x = element_text(lineheight = 0.9))
save_main_figure(fig, "supp6", 55, 70)
options(repr.plot.width = 55 / 25.4, repr.plot.height = 70 / 25.4)
print(fig)
